In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

| Library              | Purpose                  |
| -------------------- | ------------------------ |
| torch                | Deep learning framework  |
| nn                   | Neural network layers    |
| optim                | Optimization algorithms  |
| torchvision.datasets | Standard datasets        |
| transforms           | Image preprocessing      |
| models               | Pretrained architectures |
| DataLoader           | Efficient batching       |


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 64
learning_rate = 0.001
epochs = 5
num_classes = 10

CUDA allows GPU acceleration.

Batch size controls training speed and memory usage.

CIFAR-10 contains 10 object classes.

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

ResNet expects 224×224 images, so CIFAR images are resized.

Transformations applied:

| Transformation       | Purpose                   |
| -------------------- | ------------------------- |
| Resize               | Match ResNet input size   |
| RandomHorizontalFlip | Data augmentation         |
| ToTensor             | Convert image to tensor   |
| Normalize            | Match ImageNet statistics |


In [ ]:
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Test data does not use augmentation to ensure fair evaluation.

In [ ]:
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


| Dataset  | Images |
| -------- | ------ |
| Training | 50,000 |
| Testing  | 10,000 |


Classes include:

airplane

automobile

bird

cat

deer

dog

frog

horse

ship

truck

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

DataLoader provides:

mini-batch training

shuffling

efficient GPU loading

In [ ]:
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


ResNet18 is a Residual Network containing 18 layers.

Residual connections allow deeper networks to train without the vanishing gradient problem.

In [ ]:
for param in model.parameters():
    param.requires_grad = False

This freezes the feature extraction layers.

Therefore:

convolution filters remain unchanged

only the classifier will learn

This significantly reduces training time.

In [ ]:
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Original ResNet18 classifier:

512 → 1000 classes

Modified classifier:

512 → 10 classes

This adapts the network to CIFAR-10 classification.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=learning_rate
)

CrossEntropyLoss is used for multi-class classification.

Adam optimizer provides adaptive learning rates.

Only model.fc parameters are optimized.

In [ ]:
for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print("Epoch:", epoch+1,
          "Loss:", running_loss/len(train_loader))

Epoch: 1 Loss: 0.8173127648089548
Epoch: 2 Loss: 0.6196241217577244
Epoch: 3 Loss: 0.5939440524105526
Epoch: 4 Loss: 0.5796562341396766
Epoch: 5 Loss: 0.5731329996796215


Each training iteration performs:

Forward pass

Compute loss

Backpropagation

Update classifier weights

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 80.68


Evaluation steps:

Disable gradient calculation

Predict class labels

Compare with true labels

Compute accuracy

The pretrained ResNet18 model successfully classified CIFAR-10 images using transfer learning. By freezing convolution layers and training only the classifier layer, the model achieved high accuracy while significantly reducing training time.

This demonstrates the effectiveness of transfer learning for image classification tasks with limited training resources.

# FINE TUNING UC Merced Land Use Dataset ---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import zipfile

In [ ]:
# 1. Device Configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. Hyperparameters

batch_size = 32
learning_rate = 1e-4   # lower LR for fine-tuning
epochs = 5
num_classes = 21


In [ ]:
# 3. Download Dataset (Kaggle)

os.environ['KAGGLE_API_TOKEN'] = "KGAT_a853f08ac110bd4ef96c1dc02b2cb806"

!pip install -q kaggle

!kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset

# unzip
with zipfile.ZipFile("uc-merced-land-use-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

Dataset URL: https://www.kaggle.com/datasets/abdulhasibuddin/uc-merced-land-use-dataset
License(s): MIT
uc-merced-land-use-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# 4. Data Transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
# 5. Dataset Loading

dataset = datasets.ImageFolder(
    root="data/UCMerced_LandUse/Images",
    transform=transform
)

# Split into train/test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 6. Load Pretrained Model

model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# 7. Fine-Tuning Strategy

# Freeze early layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze LAST BLOCK (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace FC layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [ ]:
# 8. Loss + Optimizer

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
])

In [ ]:
# 9. Training Loop

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/5], Loss: 1.1129
Epoch [2/5], Loss: 0.1668
Epoch [3/5], Loss: 0.0621
Epoch [4/5], Loss: 0.0443
Epoch [5/5], Loss: 0.0221


In [ ]:
# 10. Evaluation

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 99.05%
